In [ ]:
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM
from sentence_transformers.util import cos_sim
import torch
import os
import json
import subprocess
from typing import List, Dict, Any
import logging

class GraphCodeBERTPatternDetector:
    def __init__(self, similarity_threshold: float = 0.60):
        print("🚀 Carregando modelo GraphCodeBERT...")
        self.tokenizer = AutoTokenizer.from_pretrained("microsoft/graphcodebert-base")
        self.model = AutoModel.from_pretrained("microsoft/graphcodebert-base")
        self.similarity_threshold = similarity_threshold

        logging.basicConfig(level=logging.INFO)
        self.logger = logging.getLogger(__name__)

        # --- Padrões conhecidos ---
        self.pattern_descriptions = {
            "Singleton": "Classe com apenas uma instância e método get_instance() ou instance().",
            "Factory Method": "Método que cria objetos de subclasses específicas.",
            "Builder": "Constrói objetos complexos passo a passo, separando a construção da representação.",
            "Adapter": "Converte interface de uma classe em outra interface esperada pelo cliente.",
            "Observer": "Define dependência um-para-muitos onde observadores são notificados automaticamente.",
            "Strategy": "Define uma família de algoritmos intercambiáveis.",
            "MVC": "Separação entre Model, View e Controller.",
            "Repository": "Interface de abstração de persistência entre domínio e banco de dados.",
            "Service": "Camada de lógica de negócio independente da interface.",
            "Clean Architecture": "Separação em camadas (Entities, Use Cases, Interface Adapters, Frameworks)."
        }

        # Pré-computar embeddings dos padrões
        self.pattern_embeddings = self._precompute_pattern_embeddings()

    def _precompute_pattern_embeddings(self):
        pattern_texts = list(self.pattern_descriptions.values())
        return torch.cat([self.get_embedding(txt) for txt in pattern_texts], dim=0)

    def get_embedding(self, code: str) -> torch.Tensor:
        try:
            tokens = self.tokenizer(
                code,
                return_tensors="pt",
                truncation=True,
                padding=True,
                max_length=512
            )
            with torch.no_grad():
                outputs = self.model(**tokens)
            return outputs.last_hidden_state.mean(dim=1)
        except Exception as e:
            self.logger.error(f"Erro ao gerar embedding: {e}")
            return torch.zeros(1, 768)

    def analyze_file(self, file_path: str) -> Dict[str, Any]:
        try:
            with open(file_path, "r", encoding="utf-8") as file:
                code = file.read()

            if len(code.strip()) < 50:
                return {"file": file_path, "pattern_detected": "Código muito curto", "similarity": 0.0}

            embedding = self.get_embedding(code)
            similarities = cos_sim(embedding, self.pattern_embeddings)[0]

            best_idx = int(torch.argmax(similarities))
            best_score = float(similarities[best_idx])
            best_pattern = list(self.pattern_descriptions.keys())[best_idx]

            return {
                "file": file_path,
                "pattern_detected": best_pattern if best_score >= self.similarity_threshold else "Nenhum",
                "similarity": round(best_score, 3)
            }

        except Exception as e:
            self.logger.error(f"Erro ao analisar {file_path}: {e}")
            return {"file": file_path, "pattern_detected": "Erro", "similarity": 0.0}

    def get_all_python_files(self, repo_name: str) -> List[str]:
        py_files = []
        for root, _, files in os.walk(repo_name):
            for f in files:
                if f.endswith(".py") and not any(x in root for x in [".git", "__pycache__", "venv", "env"]):
                    py_files.append(os.path.join(root, f))
        return py_files

    def detect_patterns(self, repo_url: str, repo_name: str = "repo") -> List[Dict[str, Any]]:
        if not self.clone_repository(repo_url, repo_name):
            return []
        files = self.get_all_python_files(repo_name)
        results = []
        for file in files:
            results.append(self.analyze_file(file))
        return results

    def clone_repository(self, repo_url: str, repo_name: str = "repo") -> bool:
        if os.path.exists(repo_name):
            return True
        result = subprocess.run(["git", "clone", repo_url, repo_name], capture_output=True, text=True)
        return result.returncode == 0

    def generate_report(self, results: List[Dict[str, Any]], output_file: str = "graphcodebert_patterns.json"):
        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(results, f, indent=4, ensure_ascii=False)
        print(f"✅ Relatório salvo em {output_file}")

# ============================================================
# 🚀 ETAPA 2 — ANÁLISE DOS RESULTADOS COM UMA LLM PÚBLICA
# ============================================================

def analyze_with_llm(results_json: str, model_name: str = "mistralai/Mistral-7B-Instruct-v0.2"):
    from transformers import AutoTokenizer, AutoModelForCausalLM

    print(f"\n🧠 Carregando modelo de linguagem: {model_name}...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto"
    )

    with open(results_json, "r", encoding="utf-8") as f:
        results = json.load(f)

    prompt = f"""
Você é um especialista em arquitetura de software.
Analise os resultados a seguir e explique quais padrões de projeto predominam,
como eles se relacionam e o que isso indica sobre a arquitetura geral.

Resultados detectados:
{json.dumps(results, indent=2, ensure_ascii=False)}

Resuma tecnicamente os principais padrões e características observadas.
"""

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=400)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    print("\n📊 Interpretação da LLM:\n")
    print(response)
    return response

# ============================================================
# 🔧 EXECUÇÃO COMPLETA
# ============================================================

if __name__ == "__main__":
    repo_url = "https://github.com/harry0703/MoneyPrinterTurbo"

    detector = GraphCodeBERTPatternDetector(similarity_threshold=0.65)
    results = detector.detect_patterns(repo_url)
    detector.generate_report(results)

    print("\n📊 Resumo:")
    print(f"Arquivos analisados: {len(results)}")
    print("Padrões detectados:")
    for r in results:
        print(f" - {os.path.basename(r['file'])}: {r['pattern_detected']} ({r['similarity']})")

    # 🔍 Analisar relatório com uma LLM pública
    analyze_with_llm("graphcodebert_patterns.json", model_name="mistralai/Mistral-7B-Instruct-v0.2")


🚀 Carregando modelo GraphCodeBERT...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/539 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at microsoft/graphcodebert-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]